# 01 - Exploratory Data Analysis
**Credit Score Classification | MLOps Group 8**

This notebook performs a thorough EDA on the raw training dataset:
- Dataset overview and shape
- Missing value analysis
- Target distribution
- Univariate distributions (numeric + categorical)
- Bivariate analysis by Credit_Score
- Correlation heatmap
- Key insights summary

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

REPO_ROOT   = Path().resolve().parent
DATA_RAW    = REPO_ROOT / 'data' / 'raw' / 'train_raw.csv'
FIGURES_DIR = REPO_ROOT / 'artifacts' / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Data path:', DATA_RAW)

## 1. Load Raw Data

In [ ]:
df = pd.read_csv(DATA_RAW, low_memory=False)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== dtypes ===')
print(df.dtypes)
print('\n=== memory usage ===')
print(df.memory_usage(deep=True).sum() / 1024**2, 'MB')

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = missing / len(df) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0]
print(f'Columns with missing values: {len(missing_df)}')
missing_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
missing_df['missing_pct'].plot(kind='barh', ax=ax, color='#e07b54')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Column')
ax.axvline(5, color='red', ls='--', alpha=0.7, label='5% threshold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missing_values.png', dpi=150)
plt.show()

## 3. Target Distribution

In [ ]:
target_counts = df['Credit_Score'].value_counts().sort_index()
target_pct    = target_counts / len(df) * 100
print('Credit_Score distribution:')
for cls, cnt, pct in zip(target_counts.index, target_counts.values, target_pct.values):
    print(f'  {cls:10s}: {cnt:7,d}  ({pct:.1f}%)')

In [ ]:
color_map = {'Poor': '#d9534f', 'Standard': '#f0ad4e', 'Good': '#5cb85c'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(target_counts.index,
                   target_counts.values,
                   color=[color_map.get(c, 'steelblue') for c in target_counts.index])
for bar, pct in zip(bars, target_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{pct:.1f}%', ha='center', fontsize=12)
axes[0].set_title('Credit Score Class Distribution')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie chart
axes[1].pie(target_counts.values,
            labels=target_counts.index,
            autopct='%1.1f%%',
            colors=[color_map.get(c, 'steelblue') for c in target_counts.index],
            startangle=140)
axes[1].set_title('Credit Score Proportions')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'target_distribution.png', dpi=150)
plt.show()

## 4. Numeric Feature Distributions

In [ ]:
# Identify numeric columns (exclude ID/target)
EXCLUDE = ['ID', 'Customer_ID', 'Name', 'SSN', 'Month', 'Credit_Score']
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in EXCLUDE]
cat_cols = [c for c in df.select_dtypes(include=['object']).columns if c not in EXCLUDE]
print(f'Numeric cols ({len(num_cols)}):', num_cols)
print(f'Categorical cols ({len(cat_cols)}):', cat_cols)

In [ ]:
print(df[num_cols].describe().T.to_string())

In [ ]:
n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    axes[i].hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Categorical Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 5))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    vc = df[col].value_counts().head(15)
    vc.plot(kind='barh', ax=ax, color='#5b8db8')
    ax.set_title(col)
    ax.set_xlabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Categorical Feature Distributions', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Bivariate Analysis: Feature vs. Credit_Score

In [ ]:
key_num_cols = [
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary',
    'Outstanding_Debt', 'Credit_Utilization_Ratio',
    'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Credit_History_Age'
]
key_num_cols = [c for c in key_num_cols if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
order = ['Poor', 'Standard', 'Good']
palette = {'Poor': '#d9534f', 'Standard': '#f0ad4e', 'Good': '#5cb85c'}

for i, col in enumerate(key_num_cols):
    sns.boxplot(
        data=df[df['Credit_Score'].isin(order)],
        x='Credit_Score', y=col, order=order,
        palette=palette, ax=axes[i], showfliers=False
    )
    axes[i].set_title(col)
    axes[i].set_xlabel('')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Key Numeric Features vs Credit Score', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bivariate_numeric.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Credit_Mix vs Credit_Score stacked bar
if 'Credit_Mix' in df.columns:
    ct = pd.crosstab(df['Credit_Mix'], df['Credit_Score'], normalize='index') * 100
    ct = ct.reindex(columns=['Poor', 'Standard', 'Good'], fill_value=0)
    ct.plot(kind='bar', stacked=True, figsize=(8, 5),
            color=['#d9534f', '#f0ad4e', '#5cb85c'])
    plt.title('Credit Mix vs Credit Score')
    plt.ylabel('Percentage (%)')
    plt.xticks(rotation=0)
    plt.legend(title='Credit Score')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'credit_mix_vs_score.png', dpi=150)
    plt.show()

## 7. Correlation Heatmap

In [ ]:
# Encode target for correlation
label_map = {'Poor': 0, 'Standard': 1, 'Good': 2}
df_corr = df[num_cols + ['Credit_Score']].copy()
df_corr['Credit_Score_enc'] = df_corr['Credit_Score'].map(label_map)
df_corr = df_corr.drop(columns=['Credit_Score'])

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Heatmap (with encoded Credit_Score)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top correlations with target
target_corr = corr_matrix['Credit_Score_enc'].drop('Credit_Score_enc').abs().sort_values(ascending=False)
print('Top correlations with Credit Score:')
print(target_corr.head(12).to_string())

## 8. Outlier Analysis

In [ ]:
# IQR-based outlier detection
outlier_report = []
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    outlier_report.append({
        'column': col, 'Q1': q1, 'Q3': q3, 'IQR': iqr,
        'lower_fence': lo, 'upper_fence': hi,
        'n_outliers': n_out, 'pct_outliers': n_out / len(df) * 100
    })

outlier_df = pd.DataFrame(outlier_report).sort_values('n_outliers', ascending=False)
print(outlier_df[['column','n_outliers','pct_outliers','lower_fence','upper_fence']].to_string(index=False))

## 9. Key Insights Summary

In [ ]:
insights = [
    f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns",
    f"Target class imbalance: Poor={target_pct.get('Poor',0):.1f}%, "
    f"Standard={target_pct.get('Standard',0):.1f}%, Good={target_pct.get('Good',0):.1f}%",
    f"Columns with missing values: {len(missing_df)}",
    f"Highest missing: {missing_df.index[0]} at {missing_df['missing_pct'].iloc[0]:.1f}%"
    if len(missing_df) > 0 else 'No missing values',
    f"Strongest predictor of Credit Score: {target_corr.index[0]} (r={target_corr.iloc[0]:.3f})",
    "Action: Use stratified 70/15/15 split and class_weight='balanced' in models.",
    "Action: Apply 99th-percentile capping for Annual_Income before training.",
    "Action: Credit_Mix and Payment_Behaviour need ordinal/OHE encoding."
]

print('=== KEY INSIGHTS ===')
for i, insight in enumerate(insights, 1):
    print(f'  {i}. {insight}')